# Phase 4C.1 Learning Curve - Colab Execution Notebook

**Repository**: nomozer/forensics-web-lab
**Branch**: research/phase-4c1-learning-curve
**Foundation**: v0.1.0-research-foundation (merge commit 575a786)
**Preregistration**: ml/configs/pilot_a_binary_preregistered.yaml (hash 727fc316...)

This notebook executes the Phase 4C.1 learning-curve experiments on Google Colab GPU.
Default mode: DRY-RUN (stops before training). Set EXECUTE=true to run training.

In [ ]:
# Cell 1: Runtime Environment Verification
import sys, os, platform, torch, torchvision, numpy, psutil, json
print(f"Python: {sys.version}")
print(f"Platform: {platform.platform()}")
print(f"PyTorch: {torch.__version__}")
print(f"TorchVision: {torchvision.__version__}")
print(f"NumPy: {numpy.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
print(f"RAM: {psutil.virtual_memory().total / 1024**3:.1f} GB")
print(f"CPU Cores: {os.cpu_count()}")

In [ ]:
# Cell 2: Google Drive Mount & Secrets
from google.colab import drive, userdata
drive.mount('/content/drive', force_remount=True)

# Read GITHUB_TOKEN from Colab Secrets (set in Colab UI: Secrets -> GITHUB_TOKEN)
try:
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
    print("GITHUB_TOKEN loaded from Colab Secrets")
except:
    GITHUB_TOKEN = os.environ.get('GITHUB_TOKEN')
    if GITHUB_TOKEN:
        print("GITHUB_TOKEN loaded from environment")
    else:
        print("WARNING: GITHUB_TOKEN not found - private repo access may fail")

In [ ]:
# Cell 3: Clone Repository at Exact Commitimport subprocess, osREPO_URL = "https://github.com/nomozer/forensics-web-lab.git"EXECUTION_CODE_REF = "2e6ce2e0fedaead45dec23b24e9d9c5a5fcf777e"  # Phase 4C.1B execution code (contains runner, validator, config)TARGET_COMMIT = EXECUTION_CODE_REFREPO_DIR = "/content/forensics-web-lab"if not os.path.exists(REPO_DIR):    # Use credential helper for authentication if GITHUB_TOKEN available    if 'GITHUB_TOKEN' in globals() and GITHUB_TOKEN:        auth_url = f"https://{GITHUB_TOKEN}@github.com/nomozer/forensics-web-lab.git"        subprocess.run(["git", "clone", auth_url, REPO_DIR], check=True)        # Reset remote URL to remove credentials        subprocess.run(["git", "remote", "set-url", "origin", REPO_URL], cwd=REPO_DIR, check=True)    else:        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)os.chdir(REPO_DIR)subprocess.run(["git", "fetch", "origin", "--prune"], check=True)subprocess.run(["git", "checkout", TARGET_COMMIT], check=True)print(f"Checked out {subprocess.run(['git', 'rev-parse', 'HEAD'], capture_output=True, text=True, check=True).stdout.strip()}")# Verify clean URLremote_url = subprocess.run(['git', 'remote', 'get-url', 'origin'], capture_output=True, text=True, cwd=REPO_DIR, check=True).stdout.strip()print(f"Remote URL: {remote_url}")assert not GITHUB_TOKEN in remote_url if 'GITHUB_TOKEN' in globals() and GITHUB_TOKEN else True, "Credentials leaked in remote URL"# Verify execution code commitactual_commit = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True, cwd=REPO_DIR, check=True).stdout.strip()if actual_commit != EXECUTION_CODE_REF:    raise RuntimeError(f"Execution code commit mismatch: expected {EXECUTION_CODE_REF}, got {actual_commit}")print(f"Execution code verified: {actual_commit}")

In [ ]:
# Cell 4: Install Dependencies (CUDA-enabled PyTorch)
!pip install --upgrade pip -q
# Install PyTorch with CUDA 12.1 (Colab default)
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121 -q
# Install other dependencies
!pip install -r ml/requirements.txt -q
print("Dependencies installed")

In [ ]:
# Cell 5: Mount Private Google Drive for Bundle & ArtifactsDRIVE_BUNDLE_DIR = "/content/drive/MyDrive/forensics-web-lab/phase_4c1/smoke_n50_seed42"EXTRACT_DIR = "/content/bundle"ARTIFACT_DEST = "/content/drive/MyDrive/forensics-web-lab/phase_4c1_artifacts/n50_seed42"os.makedirs(BUNDLE_DEST, exist_ok=True)os.makedirs(ARTIFACT_DIR, exist_ok=True)ARCHIVE_PATH = os.path.join(DRIVE_BUNDLE_DIR, "phase_4c1_smoke_n50_seed42.tar")SHA_PATH = os.path.join(DRIVE_BUNDLE_DIR, "phase_4c1_smoke_n50_seed42.tar.sha256")if os.path.exists(ARCHIVE_PATH) and os.path.exists(SHA_PATH):    import shutil    print(f"Verifying archive at {ARCHIVE_PATH}...")    import hashlib    print(f"Computing SHA-256 of archive...")    with open(ARCHIVE_PATH, "rb") as f:        archive_sha256 = hashlib.sha256(f.read()).hexdigest()    with open(SHA_PATH, "r") as f:        expected_sha256 = f.read().strip()    if archive_sha256 != expected_sha256:        raise RuntimeError(f"Archive SHA-256 mismatch: {archive_sha256} != {expected_sha256}")    print(f"Archive SHA-256 verified: {archive_sha256}")    print(f"Extracting archive to {EXTRACT_DIR}...")    import tarfile    with tarfile.open(ARCHIVE_PATH, "r") as tar:        tar.extractall(EXTRACT_DIR)    print("Archive extracted successfully")        src = os.path.join(BUNDLE_SOURCE, f)        dst = os.path.join(BUNDLE_DEST, f)        if os.path.isdir(src):            shutil.copytree(src, dst, dirs_exist_ok=True)        else:            shutil.copy2(src, dst)    print("Bundle copied successfully")else:    print(f"WARNING: Bundle not found at {BUNDLE_SOURCE}")    print("Run ml/datasets/export_phase_4c1_bundle.py --execute locally and upload to Drive first")

In [ ]:
# Cell 6: Verify Bundle Integrityimport syssys.path.insert(0, "/content/forensics-web-lab")from ml.datasets.validate_phase_4c1_bundle import main as validate_bundleimport syssys.argv = ["validate", "--bundle", "/content/bundle", "--smoke-only"]validate_bundle()

In [ ]:
# Cell 7: Dry-Run Experiment Matrix
import yaml
with open("ml/configs/phase_4c1_learning_curve.yaml") as f:
    config = yaml.safe_load(f)

print("=" * 60)
print("PHASE 4C.1 EXPERIMENT MATRIX (DRY-RUN)")
print("=" * 60)
print(f"Sample sizes: {[s['n_sources'] for s in config['learning_curve']['sample_sizes'] if s['status']=='runnable']}")
print(f"Seeds: {config['hyperparameters']['seeds']}")
print(f"Stage 1 runs: {len([s for s in config['learning_curve']['sample_sizes'] if s['status']=='runnable'])} sizes × {len(config['hyperparameters']['seeds'])} seeds = {len([s for s in config['learning_curve']['sample_sizes'] if s['status']=='runnable']) * len(config['hyperparameters']['seeds'])} runs")
print(f"Baselines: {len(config['baselines'])}")
print(f"Baseline runs: {len([s for s in config['learning_curve']['sample_sizes'] if s['status']=='runnable']) * len(config['hyperparameters']['seeds']) * len(config['baselines'])} runs")
print(f"Stage 2: CONDITIONAL (gated by Stage 1 > Dummy AND > Metadata-only)")
print(f"Locked test: SEALED (0 evaluations)")

# Verify dataset manifest
import pandas as pd
manifest = pd.read_csv("/content/bundle/manifest_pilot_a_option_p.csv")
print(f"\nManifest rows: {len(manifest)}")
print(f"Partitions: {manifest['partition'].value_counts().to_dict()}")
print(f"Nested invariant N50⊂N100⊂N250: {(manifest['lc_n50']=='True').sum()}=50, {(manifest['lc_n100']=='True').sum()}=100, {(manifest['lc_n250']=='True').sum()}=250")
print(f"Locked-test sources in manifest: {(manifest['partition']=='locked_test').sum()} (will be filtered)")

In [ ]:
# Cell 8: EXECUTION GATE - Set to True to Enable Training
EXECUTE = False  # <<< SET TO True TO RUN TRAINING

if not EXECUTE:
    print("=" * 60)
    print("DRY-RUN MODE - Training NOT executed")
    print("Set EXECUTE = True to run training")
    print("=" * 60)
else:
    print("=" * 60)
    print("EXECUTION MODE - Training WILL run")
    print("=" * 60)

In [ ]:
# Cell 9: GPU Gate (conditional on EXECUTE)
if EXECUTE:
    import torch
    assert torch.cuda.is_available(), "COLAB_GPU_REQUIRED: CUDA not available. Switch runtime to T4 GPU."
    assert torch.cuda.device_count() >= 1, "COLAB_GPU_REQUIRED: No CUDA devices found."
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    
    # Check available disk space
    import shutil
    free_space = shutil.disk_usage("/content").free / 1024**3
    print(f"Free disk space: {free_space:.1f} GB")
    assert free_space >= 10, "Insufficient disk space: need at least 10 GB"
    
    # Check Drive space
    drive_free = shutil.disk_usage("/content/drive").free / 1024**3
    print(f"Drive free space: {drive_free:.1f} GB")
    assert drive_free >= 5, "Insufficient Drive space: need at least 5 GB"
else:
    print("Skipped - EXECUTE=False (CPU dry-run permitted)")

In [ ]:
# Cell 10: Stage 1 Training Loop (Conditional on EXECUTE)
if EXECUTE:
    import sys
    sys.path.insert(0, '/content/forensics-web-lab')
    from ml.training.run_phase_4c1 import main as run_training
    import sys
    sys.argv = [
        "run_phase_4c1",
        "--config", "ml/configs/phase_4c1_learning_curve.yaml",
        "--bundle", "/content/bundle",
        "--sample-size", "50",
        "--seed", "42",
        "--stage", "frozen",
        "--device", "cuda",
        "--train-partition", "development_train",
        "--eval-partition", "inner_validation",
        "--output", "/content/phase_4c1_artifacts/n50_seed42",
    ]
    exit_code = 0
    try:
        run_training()
    except SystemExit as e:
        exit_code = e.code
    except Exception as e:
        print(f"ERROR: {e}")
        import traceback
        traceback.print_exc()
        exit_code = 1

    if exit_code != 0:
        raise RuntimeError(f"Training failed with exit code {exit_code}")
else:
    print("Skipped - EXECUTE=False")

In [ ]:
# Cell 11: Artifact Verification
if EXECUTE:
    import json, hashlib, os
    artifact_dir = '/content/phase_4c1_artifacts/n50_seed42'
    
    # Check required artifacts exist
    required_files = [
        'run_receipt.json',
        'best_checkpoint.pt',
        'predictions.json',
        'epoch_history.json',
    ]
    missing = []
    for f in required_files:
        if not os.path.exists(os.path.join(artifact_dir, f)):
            missing.append(f)
    
    if missing:
        raise RuntimeError(f"Missing artifacts: {missing}")
    
    # Verify run receipt
    with open(os.path.join(artifact_dir, 'run_receipt.json'), 'r') as f:
        receipt = json.load(f)
    
    assert receipt['status'] == 'completed', f"Run not completed: {receipt['status']}"
    assert receipt['locked_test_access'] == 0, f"Locked test accessed: {receipt['locked_test_access']}"
    
    # Verify checkpoint
    checkpoint_path = os.path.join(artifact_dir, 'best_checkpoint.pt')
    assert os.path.exists(checkpoint_path), f'Checkpoint not found: {checkpoint_path}'
    
    # Verify checkpoint SHA-256
    with open(checkpoint_path, 'rb') as f:
        actual_sha256 = hashlib.sha256(f.read()).hexdigest()
    expected_sha256 = receipt.get('checkpoint_sha256', '')
    assert actual_sha256 == expected_sha256, f'Checkpoint SHA mismatch: {actual_sha256} != {expected_sha256}'
    
    print(f"Checkpoint SHA-256 verified: {actual_sha256}")
    print(f"Locked test access: {receipt['locked_test_access']}")
    print(f"Training time: {receipt['training_time_seconds']:.1f}s")
    print(f"Peak VRAM: {receipt['peak_vram_mb']:.1f} MB")
    print(f"Best Macro-F1: {receipt['best_val_macro_f1']:.4f}")
else:
    print("Skipped - EXECUTE=False")

In [ ]:
# Cell 12: Stage 2 Eligibility Report
if EXECUTE:
    import json
    with open('/content/phase_4c1_artifacts/n50_seed42/run_receipt.json') as f:
        receipt = json.load(f)
    
    stage1_macro_f1 = receipt.get('best_val_macro_f1', 0)
    dummy_result = receipt.get('dummy_baseline', {})
    meta_result = receipt.get('metadata_baseline', {})
    
    dummy_f1 = dummy_result.get('macro_f1_mean', 0)
    meta_f1 = meta_result.get('macro_f1_mean', 0)
    
    print(f"Stage 1 Macro-F1: {stage1_macro_f1:.4f}")
    print(f"Dummy Baseline: {dummy_f1:.4f}")
    print(f"Metadata Baseline: {meta_f1:.4f}")
    
    if stage1_macro_f1 > dummy_f1 and stage1_macro_f1 > meta_f1:
        print("ELIGIBLE: Stage 1 > Dummy AND > Metadata-only")
        print("Stage 2 fine-tuning may proceed (requires separate approval)")
    else:
        if stage1_macro_f1 <= dummy_f1 and stage1_macro_f1 <= meta_f1:
            print("NOT_ELIGIBLE: Stage 1 <= Dummy AND <= Metadata-only")
        elif stage1_macro_f1 <= dummy_f1:
            print("NOT_ELIGIBLE: Stage 1 <= Dummy")
        else:
            print("NOT_ELIGIBLE: Stage 1 <= Metadata-only")
        print("Stage 2 fine-tuning blocked")
else:
    print("Skipped - EXECUTE=False")

In [ ]:
# Cell 13: Final Summary
print("=" * 60)
print("PHASE 4C.1 EXECUTION COMPLETE")
print("=" * 60)
if not EXECUTE:
    print("Mode: DRY-RUN")
    print("No training executed")
    print("To run: Set EXECUTE = True in Cell 8 and re-run")
else:
    print("Mode: EXECUTION")
    
    # Verify all required artifacts
    import os
    artifact_dir = '/content/phase_4c1_artifacts/n50_seed42'
    required = ['run_receipt.json', 'best_checkpoint.pt', 'predictions.json', 'epoch_history.json']
    all_exist = all(os.path.exists(os.path.join(artifact_dir, f)) for f in required)
    
    if all_exist:
        print("Training artifacts saved to Google Drive")
        print("Run ID: phase4c1-stage1-n50-seed42-colab")
        print(f"Run receipt: {artifact_dir}/run_receipt.json")
        print(f"Checkpoint: {artifact_dir}/best_checkpoint.pt")
        print("Next: Stage 2 gate evaluation (separate approval)")
        print("Next: Statistical analysis + Locked-test evaluation (separate approval)")
    else:
        print("ERROR: Some artifacts missing")
        missing = [f for f in required if not os.path.exists(os.path.join(artifact_dir, f))]
        print(f"Missing: {missing}")

In [ ]:
# Cell 14: GPU Environment Info (for evidence)
import torch
import platform
import json

gpu_info = {
    "cuda_available": torch.cuda.is_available(),
    "device_count": torch.cuda.device_count() if torch.cuda.is_available() else 0,
    "device_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "cuda_version": torch.version.cuda if torch.cuda.is_available() else None,
    "pytorch_version": torch.__version__,
    "platform": platform.platform(),
    "python_version": platform.python_version(),
}

print(json.dumps(gpu_info, indent=2))